### TEST · dominio pequeño · VAR con coeficientes por bloque del año (estaciones o meses)
El PVAR con armónicos (h1b, h2b) da la mitad de picos de f en marzo–mayo y septiembre–octubre. Con un VAR por estación (`e4b`) se arregla marzo pero no abril–mayo ni septiembre–octubre: en la dirección de temporal la σ simulada sigue un 11–17 % baja.
Aquí el VAR puede tener coeficientes distintos por **estación** (`e4b`) o por **mes** (`m12b`). p = 10, choques con bootstrap mensual. Se elige con `ESQUEMA` y se guarda como `var_{NOMBRE}.pkl` y `simulaciones_{NOMBRE}/` en `outputs/var_6h_wmed/`; luego 03 y 04 con ese nombre.

In [1]:
import os, pickle, time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils.tv_var.periodic_var import PeriodicVAR

#### Load Data

In [2]:
out = 'outputs/var_6h_wmed'

pcs_sel_6h = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)
harm_std_pcs = pickle.load(open(f'{out}/harm_std_pcs.pkl', 'rb'))          # el mismo que el 02A
pcs_sel_6h_z = harm_std_pcs.transform(pcs_sel_6h)

P = 10
ESQUEMA = 'meses'                              # 'estaciones' (e4b) o 'meses' (m12b)

if ESQUEMA == 'estaciones':
    NOMBRE, BLOQUES = 'e4b', ['DJF', 'MAM', 'JJA', 'SON']
    est_de = lambda m: np.select([np.isin(m, [12, 1, 2]), np.isin(m, [3, 4, 5]), np.isin(m, [6, 7, 8])], [0, 1, 2], 3)
else:
    NOMBRE, BLOQUES = 'm12b', ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']
    est_de = lambda m: np.asarray(m) - 1
N_B = len(BLOQUES)
print(NOMBRE, N_B, 'bloques')

m12b 12 bloques


#### Ajuste: un VAR por estación (MCO sobre la misma matriz de lags que PeriodicVAR)

In [3]:
X_lag, Y = PeriodicVAR(p=P, n_harm=0, freq='6h')._lag_matrix(pcs_sel_6h_z, P)
print(X_lag.columns[:3].tolist(), X_lag.columns[30:32].tolist())     # const, PC1_lag_1, …; PC30_lag_1, PC1_lag_2

est = est_de(X_lag.index.month)
k = Y.shape[1]
coefs = {}
resid = np.empty(Y.shape)
for e in range(N_B):
    sel = est == e
    coefs[e] = np.linalg.lstsq(X_lag.values[sel], Y.values[sel], rcond=None)[0]       # (1 + k·p, k)
    resid[sel] = Y.values[sel] - X_lag.values[sel] @ coefs[e]
    print(f'{BLOQUES[e]}: {sel.sum()} filas')

resid = pd.DataFrame(resid, index=Y.index, columns=Y.columns)
resid_mes = {m: resid[resid.index.month == m].values for m in range(1, 13)}

['const', 'msl_PC1_lag_1', 'msl_PC2_lag_1'] ['msl_PC30_lag_1', 'msl_PC1_lag_2']
ene: 10778 filas
feb: 9832 filas
mar: 10788 filas
abr: 10440 filas
may: 10788 filas
jun: 10440 filas
jul: 10788 filas
ago: 10788 filas
sep: 10440 filas
oct: 10788 filas
nov: 10320 filas
dic: 10664 filas


In [4]:
#### Estabilidad de cada bloque: radio espectral de la matriz compañera (< 1)
for e in range(N_B):
    C = np.zeros((k * P, k * P))
    C[:k] = coefs[e][1:].T
    C[k:, :-k] = np.eye(k * (P - 1))
    print(f'{BLOQUES[e]}: radio espectral {np.abs(np.linalg.eigvals(C)).max():.4f}')

ene: radio espectral 0.9982
feb: radio espectral 0.9989
mar: radio espectral 0.9991
abr: radio espectral 0.9992
may: radio espectral 0.9997
jun: radio espectral 1.0003
jul: radio espectral 0.9996
ago: radio espectral 0.9982
sep: radio espectral 0.9956
oct: radio espectral 0.9949
nov: radio espectral 0.9953
dic: radio espectral 0.9960


#### Simulación (bootstrap de los residuos de cada mes)

In [5]:
def simular(seed, fechas, burn):
    rng = np.random.default_rng(seed)
    fechas_t = pd.date_range(fechas[0] - pd.Timedelta('6h') * burn, periods=len(fechas) + burn, freq='6h')
    e_t = est_de(fechas_t.month)
    eps = np.empty((len(fechas_t), k))
    for mes, pool in resid_mes.items():
        sel = fechas_t.month == mes
        eps[sel] = pool[rng.integers(len(pool), size=sel.sum())]
    z = np.zeros((P + len(fechas_t), k))
    for s in range(len(fechas_t)):
        t = P + s
        fila = np.concatenate([[1.0], z[t - P:t][::-1].ravel()])          # const, lag 1 (todas las PCs), lag 2, …
        z[t] = fila @ coefs[e_t[s]] + eps[s]
    return pd.DataFrame(z[P + burn:], index=fechas, columns=Y.columns)

burn = int(pd.Timedelta(days=3 * 365) / pd.Timedelta('6h'))
fechas = pd.date_range('1940-01-01', '2026-12-31 18:00', freq='6h')

pickle.dump({'coefs': coefs, 'resid_mes': resid_mes, 'p': P, 'cols': list(Y.columns), 'esquema': ESQUEMA}, open(f'{out}/var_{NOMBRE}.pkl', 'wb'))
sim_dir = f'{out}/simulaciones_{NOMBRE}'
os.makedirs(sim_dir, exist_ok=True)
for i in range(20):
    f_sim = f'{sim_dir}/sim_{i:03d}.parquet'
    if os.path.exists(f_sim):
        continue
    t0 = time.time()
    harm_std_pcs.inverse_transform(simular(i, fechas, burn)).astype('float32').to_parquet(f_sim)
    print(f'{NOMBRE} sim {i:03d}  ({time.time() - t0:.0f} s)')

m12b sim 000  (6 s)
m12b sim 001  (5 s)
m12b sim 002  (5 s)
m12b sim 003  (5 s)
m12b sim 004  (5 s)
m12b sim 005  (6 s)
m12b sim 006  (6 s)
m12b sim 007  (6 s)
m12b sim 008  (5 s)
m12b sim 009  (6 s)
m12b sim 010  (6 s)
m12b sim 011  (6 s)
m12b sim 012  (6 s)
m12b sim 013  (6 s)
m12b sim 014  (5 s)
m12b sim 015  (6 s)
m12b sim 016  (6 s)
m12b sim 017  (6 s)
m12b sim 018  (5 s)
m12b sim 019  (5 s)


#### Validación rápida: σ por mes y autocorrelación a 1 día por mes, variantes vs observado

In [8]:
import glob

def cargar(nombre, n=20):
    return [harm_std_pcs.transform(pd.read_parquet(f))[Y.columns]
            for f in sorted(glob.glob(f'{out}/simulaciones_{nombre}/sim_*.parquet'))[:n]]

def std_mes(z):
    return z.groupby(z.index.month).std().mean(axis=1)

def acf1d_mes(z, l=4):
    v = z.values; mes = z.index.month[:-l]; a, b = v[:-l], v[l:]
    return pd.Series([np.nanmean([np.corrcoef(a[mes == m][:, j], b[mes == m][:, j])[0, 1] for j in range(v.shape[1])])
                      for m in range(1, 13)], index=range(1, 13))

z_obs = pcs_sel_6h_z[Y.columns]
sims = {n: cargar(n) for n in ['h1b', 'h2b','e4b', 'm12b']}

tabla = pd.DataFrame({'σ obs': std_mes(z_obs), 'ACF 1 d obs': acf1d_mes(z_obs)})
for n, lista in sims.items():
    tabla[f'σ {n}/obs'] = pd.concat([std_mes(z) for z in lista], axis=1).median(axis=1) / tabla['σ obs']
    tabla[f'ACF {n} − obs'] = pd.concat([acf1d_mes(z) for z in lista], axis=1).median(axis=1) - tabla['ACF 1 d obs']
tabla.round(3)

,σ obs,ACF 1 d obs,σ h1b/obs,ACF h1b − obs,σ h2b/obs,ACF h2b − obs,σ e4b/obs,ACF e4b − obs,σ m12b/obs,ACF m12b − obs
1,1.000,0.306,1.027,-0.003,0.996,0.001,0.991,0.002,1.001,-0.002
2,0.998,0.295,1.032,-0.011,1.007,0.002,1.018,0.013,1.002,-0.003
3,0.999,0.304,0.984,-0.020,0.991,-0.007,0.999,0.002,0.998,-0.001
4,1.006,0.303,0.965,0.002,1.003,0.000,0.986,0.011,0.999,-0.002
5,0.991,0.347,0.990,0.011,1.009,0.001,1.006,-0.022,1.000,0.001
6,1.004,0.411,1.020,0.008,0.999,0.001,0.995,0.003,1.001,-0.000
7,1.003,0.441,1.028,0.014,0.994,0.008,0.977,-0.016,0.999,-0.003
8,0.994,0.442,1.028,0.003,1.013,0.002,1.010,-0.014,1.002,-0.004
9,1.005,0.398,0.992,-0.002,1.002,0.004,1.019,-0.025,0.998,-0.001
10,0.999,0.365,0.970,-0.013,0.993,-0.011,0.986,-0.012,0.999,-0.000


In [7]:
m_obs = z_obs.groupby(z_obs.index.month).mean()
m_sim = pd.concat([z.groupby(z.index.month).mean() for z in sims['h1b']]).groupby(level=0).mean()
dif = m_sim - m_obs
print(dif.abs().max(axis=1).round(3))
dif.round(2)

1     0.023
2     0.015
3     0.030
4     0.028
5     0.041
6     0.025
7     0.027
8     0.043
9     0.025
10    0.017
11    0.014
12    0.018
dtype: float64


,msl_PC1,msl_PC2,msl_PC3,msl_PC4,msl_PC5,msl_PC6,msl_PC7,msl_PC8,msl_PC9,msl_PC10,...,msl_PC21,msl_PC22,msl_PC23,msl_PC24,msl_PC25,msl_PC26,msl_PC27,msl_PC28,msl_PC29,msl_PC30
1,-0.00,0.02,0.01,-0.00,0.01,0.01,0.01,0.01,0.00,0.00,...,0.00,-0.01,0.00,0.00,-0.01,0.01,0.01,-0.01,-0.01,0.01
2,0.01,0.01,0.02,-0.01,-0.00,-0.01,0.00,0.01,0.00,0.01,...,-0.01,0.01,0.00,0.00,-0.01,-0.00,0.01,0.00,-0.00,-0.00
3,-0.03,0.01,-0.01,0.02,0.01,-0.02,0.00,0.00,-0.00,-0.01,...,0.00,0.00,0.01,-0.00,-0.01,0.00,-0.01,-0.01,0.01,-0.00
4,-0.03,-0.01,0.03,0.02,-0.01,0.00,-0.00,-0.00,-0.01,-0.01,...,0.00,0.00,0.00,-0.00,-0.01,0.00,-0.00,-0.01,0.01,-0.01
5,-0.04,0.01,0.02,0.01,-0.00,0.02,-0.01,-0.00,-0.01,-0.01,...,-0.01,0.00,0.01,0.02,-0.01,0.01,0.01,-0.00,-0.00,-0.01
6,0.02,-0.00,0.00,0.02,0.03,-0.02,-0.01,0.00,0.01,0.00,...,0.01,0.00,0.01,-0.00,-0.01,0.00,-0.02,0.00,-0.01,0.01
7,-0.03,-0.00,0.01,0.03,-0.02,0.01,-0.01,0.01,0.00,-0.00,...,-0.00,-0.01,-0.01,-0.01,0.01,0.00,-0.01,0.01,0.02,0.01
8,0.04,-0.01,0.01,-0.02,0.02,0.00,0.02,-0.01,0.00,-0.02,...,-0.01,0.02,0.01,0.01,-0.01,-0.00,0.02,-0.01,-0.02,-0.01
9,0.02,-0.01,-0.02,0.01,-0.02,0.01,-0.01,0.01,-0.01,0.02,...,-0.00,-0.02,-0.01,-0.02,0.02,0.00,-0.02,0.00,0.01,-0.00
10,0.02,0.00,0.00,0.00,0.01,-0.00,0.01,-0.00,-0.00,-0.01,...,0.00,-0.01,-0.01,-0.00,-0.00,0.01,-0.02,-0.00,-0.01,0.01
